# Symbolize - Type Theory Logic - New Style (term-based core)

Cell for cell the same as *Type Theory - Logic V2.ipynb*, on `symbolize.terms`. Where the original differs from the new API the cell says so.

In [1]:
import sys
sys.path.insert(1, "../..")
from symbolize.terms.derive import (
    Argument, DerivationError, set_var, family, hyp,
    pair, inl, inr, cases, forall, exists,
)
from symbolize.terms import App, transform, library as L

A, B, C = set_var("A"), set_var("B"), set_var("C")
Falsum = L.Falsum

## Conjunction ##
### Introduction ###

In [2]:
p = A.get_proof('p')
q = B.get_proof('q')
r = pair(p, q)
Argument([p, q], r)

p : A   q : B
--------------
(p, q) : A ∧ B

### Elimination ###

In [3]:
Argument([r], r.fst)

(p, q) : A ∧ B
---------------
fst((p, q)) : A

In [4]:
A_and_B = A & B
r = A_and_B.get_proof('r')
Argument([r], r.fst)

r : A ∧ B
----------
fst(r) : A

In [5]:
r.snd

snd(r) : B

#### Example - From $A \land B$ deduce $B \land A$ - [ST] p72 ####

In [6]:
Argument([
    Argument([r], r.fst),
    Argument([r], r.snd)
],
    pair(r.snd, r.fst)
)

r : A ∧ B
----------
fst(r) : A   r : A ∧ B
----------
snd(r) : B
-----------------------------------------------------------------
(snd(r), fst(r)) : B ∧ A

## Implication ##
### Introduction ###

In [7]:
x = A.get_proof('x')
e = B.get_proof('e')
Argument([e], e.abstract(x), discharges=[x], label=r"\Rightarrow{I}")

[x : A] ⋮ e : B
--------------- (\Rightarrow{I})
λ(x).e : A ⟹ B

### Elimination ###

In [8]:
A_implies_B = A >> B
q = A_implies_B.get_proof('q')
a = A.get_proof('a')
q(a)

apply(q, a) : B

## Disjunction ##
### Introduction ###

In [9]:
q = A.get_proof('q')
inl(q, B)

inl(q) : ∨(A, B)

In [10]:
r = B.get_proof('r')
inr(r, A)

inr(r) : ∨(A, B)

In [11]:
B.term.arity

0

#### Example - Identity - [ST] p83 ####

In [12]:
Argument([x], x.abstract(x), discharges=[x], label=r"\Rightarrow{I}")

[x : A] ⋮ x : A
--------------- (\Rightarrow{I})
λ(x).x : A ⟹ A

#### Example - From $A \Rightarrow B$ and $B \Rightarrow C$ deduce $A \Rightarrow C$ - [ST] p83-84 ####

In [13]:
A_implies_B = A >> B
B_implies_C = B >> C
a = A_implies_B.get_proof('a')
b = B_implies_C.get_proof('b')
x = A.get_proof('x')

In [14]:
Argument([x, a], a(x))

x : A   a : A ⟹ B
-----------------
a(x) : B

In [15]:
arg1 = Argument([x, a], a(x))
arg2 = Argument([arg1, b], b(arg1.conclusion))
arg3 = Argument([arg2, x], arg2.conclusion.abstract(x))
arg4 = Argument([arg2, x], arg3.conclusion.abstract(b))
arg5 = Argument([arg2, x], arg4.conclusion.abstract(a))
arg5

x : A   a : A ⟹ B
-----------------
a(x) : B   b : B ⟹ C
--------------------------------------------------------
b(a(x)) : C   x : A
-------------------------------------------------------------------------------------------------------------------------------------
λ(a).λ(b).λ(x).(b(a(x))) : (A ⟹ B) ⟹ ((B ⟹ C) ⟹ (A ⟹ C))

In [16]:
arg5.conclusion.type.subst(C.term, Falsum)

⟹(⟹(A, B), ⟹(⟹(B, ⊥), ⟹(A, ⊥)))

In [17]:
foo = arg5.conclusion.type.subst(C.term, Falsum)

def my_sub(t, depth):
    """rewrite  X ⟹ ⊥  as  ¬X"""
    if isinstance(t, App) and t.fn == L.implies and t.args[1] == Falsum:
        return L.not_(transform(t.args[0], my_sub, depth))
    return None

foo = transform(foo, my_sub)
foo

⟹(⟹(A, B), ⟹(¬(B), ¬(A)))

#### Example - $((A \lor B) \Rightarrow C) \iff ((A \Rightarrow C) \land (B \Rightarrow C))$ - [ST] p86-87 ####
##### $((A \lor B) \Rightarrow C) \Rightarrow ((A \Rightarrow C) \land (B \Rightarrow C))$ #####

In [18]:
A_or_B_implies_C = (A | B) >> C
y = A_or_B_implies_C.get_proof('y')
x = A.get_proof('x')
w = B.get_proof('w')

In [19]:
arg1 = Argument([x], inl(x, B))
arg2 = Argument([arg1, y], y(arg1.conclusion))
arg3 = Argument([arg2], arg2.conclusion.abstract(x))

arg4 = Argument([w], inr(w, A))
arg5 = Argument([arg4, y], y(arg4.conclusion))
arg6 = Argument([arg5], arg5.conclusion.abstract(w))

arg7 = Argument([arg3, arg6], pair(arg3.conclusion, arg6.conclusion))
arg7

x : A
--------------
inl(x) : A ∨ B   y : (A ∨ B) ⟹ C
-----------------------------------------------------
y(inl(x)) : C
-------------------------------------------------------------------------------------------------------------------------
λ(x).(y(inl(x))) : A ⟹ C   w : B
--------------
inr(w) : A ∨ B   y : (A ∨ B) ⟹ C
-----------------------------------------------------
y(inr(w)) : C
-------------------------------------------------------------------------------------------------------------------------
λ(w).(y(inr(w))) : B ⟹ C
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [20]:
arg7.conclusion.abstract(y)

λ((y)pair(λ((x)apply(y, inl(x))), λ((w)apply(y, inr(w))))) : ⟹(⟹(∨(A, B), C), ∧(⟹(A, C), ⟹(B, C)))

##### $((A \lor B) \Rightarrow C) \Leftarrow ((A \Rightarrow C) \land (B \Rightarrow C))$ #####

In [21]:
A_or_B = A | B
A_implies_C_and_B_implies_C = (A >> C) & (B >> C)
z = A_or_B.get_proof('z')
p = A_implies_C_and_B_implies_C.get_proof('p')

In [22]:
p

p : ∧(⟹(A, C), ⟹(B, C))

In [23]:
Argument([z, p.select(0).alias("fst~p"), p.select(1).alias("snd~p")],
  cases(z, p.select(0).alias("fst~p"), p.select(1).alias("snd~p"))
)

z : A ∨ B   fst~p : A ⟹ C   snd~p : B ⟹ C
-----------------------------------------
cases(z, fst~p, snd~p) : C

In [24]:
cases(z, p.select(0).alias("fst~p"), p.select(1).alias("snd~p")).abstract(z).abstract(p)

λ((p)λ((z)cases(z, fst(p), snd(p)))) : ⟹(∧(⟹(A, C), ⟹(B, C)), ⟹(∨(A, B), C))

## Universal Quantifier ##
### Introduction ###

`P` is now a genuine family over `A` (`family('P', A)`) rather than a symbol with `assume_contains=[x]`.

In [25]:
x = A.get_proof('x')
P = family('P', A)
P

P(z) set [z ∈ A]

The original cell generalised a *hypothesis* `p : P(x)` over `x`. That breaks the side condition of $\forall{I}$ ([ST] p89: $x$ must not be free in any undischarged assumption), and the new core rejects it:

In [26]:
p = P(x).get_proof('p')
try:
    p.abstract(x)
except DerivationError as e:
    print(e)

Cannot discharge x: the hypothesis p depends on it


Discharging `p` first gives a valid $\forall{I}$:

In [27]:
Argument([p], p.abstract(p).abstract(x), discharges=[p, x], label=r"\forall{I}")

[p : P(x), x : A] ⋮ p : P(x)
------------------------------ (\forall{I})
λ(x).λ(p).p : ∀x.(P(x) ⟹ P(x))

### Elimination ###

In [28]:
a = A.get_proof('a')
forall_x_P = forall(x, P(x))
f = forall_x_P.get_proof('f')

Argument([a, f], f.apply(a), label=r"\forall{E}")

a : A   f : ∀x.P(x)
------------------- (\forall{E})
f(a) : P(a)

## Existential Quantifier ##
### Introduction ###

`pair(a, p)` recognises that the type of `p` depends on the witness `a`, so no `exists_expression=` hint is needed.

In [29]:
a = A.get_proof('a')
P = family('P', A)
p = P(a).get_proof('p')
Argument([a, p], pair(a, p), label=r"\exists{I}")

a : A   p : P(a)
---------------- (\exists{I})
(a, p) : ∃a.P(a)

### Elimination ###

In [30]:
x = A.get_proof('x')
P = family('P', A)

exists_x_P = exists(x, P(x))
p = exists_x_P.get_proof('p')

Argument([p], p.fst, label=r"\exists{E_1}")

p : ∃x.P(x)
----------- (\exists{E_1})
fst(p) : A

In [31]:
Argument([p], p.snd, label=r"\exists{E_2}")

p : ∃x.P(x)
------------------ (\exists{E_2})
snd(p) : P(fst(p))

### [ST] p92 ###

In [32]:
x = A.get_proof('x')
B = family('B', A)
C = family('C', A)
B_implies_C = B(x) >> C(x)
forall_x_B_implies_C = forall(x, B_implies_C)
r = forall_x_B_implies_C.get_proof('r')
forall_x_B = forall(x, B(x))
p = forall_x_B.get_proof('p')

In [33]:
Argument([r, x], r.apply(x), label=r"\forall{E}")

r : ∀x.(B(x) ⟹ C(x))   x : A
---------------------------- (\forall{E})
r(x) : B(x) ⟹ C(x)

In [34]:
Argument([x, p], p.apply(x), label=r"\forall{E}")

x : A   p : ∀x.B(x)
------------------- (\forall{E})
p(x) : B(x)

The original notebook stops here; the derivation completes as:

In [35]:
s = r(x)(p(x)).abstract(x)
Argument([Argument([r, x], r(x)), Argument([x, p], p(x))], s, discharges=[x], label=r"\forall{I}")

[x : A] ⋮ r : ∀x.(B(x) ⟹ C(x))   x : A
----------------------------
r(x) : B(x) ⟹ C(x)   x : A   p : ∀x.B(x)
-------------------
p(x) : B(x)
-------------------------------------------------------------------------------------------------------------------------------------------- (\forall{I})
λ(x).(r(x)(p(x))) : ∀x.C(x)

In [36]:
s.abstract(p).abstract(r)

λ((r)λ((p)λ((x)apply(apply(r, x), apply(p, x))))) : ⟹(∀(A, (x)⟹(B(x), C(x))), ⟹(∀(A, (x)B(x)), ∀(A, (x)C(x))))